# Rotokare Acoustic Imaging Demo
This notebook demonstrates how to load audio data, define a microphone array, and generate an acoustic "image" (likelihood map) of a sound source.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# Add src to path so imports work
sys.path.append(os.path.abspath(os.path.join('..')))

from src.micarray import MicArray
from src.audio import AudioRecording
from src.scanner import BirdScanner

## 1. Setup Microphone Array
Load the 3D coordinates of the microphones.

In [ ]:
# Example Mic positions (replace with loadtxt if you have the file)
# mic_positions = np.loadtxt('../data/metadata/mic_locations.txt')

# Mock data for demo purposes (8 mics in a rough circle)
mic_positions = np.array([
    [10, 0, 0], [7, 7, 0.5], [0, 10, 1], [-7, 7, 0.2],
    [-10, 0, 0], [-7, -7, 1.5], [0, -10, 0], [7, -7, 0.5]
])

fs = 48000
array = MicArray(mic_positions, fs=fs)

print(f"Loaded array with {len(mic_positions)} microphones.")

## 2. define Search Grid
We create a 2D grid on the ground (Z=0) to scan for sources.

In [ ]:
xmin, xmax = -20, 20
ymin, ymax = -20, 20
res = 0.5 # meters

x = np.arange(xmin, xmax, res)
y = np.arange(ymin, ymax, res)
X, Y = np.meshgrid(x, y)

print(f"Grid shape: {X.shape}")

## 3. Load Audio
We load a specific segment of audio to process.

In [ ]:
# Replace with actual path
audio_path = '../data/samples/Tui01.wav'

if os.path.exists(audio_path):
    rec = AudioRecording(audio_path, lazy=True)
    # Get a 0.5s chunk from the middle
    chunk = rec.get_window(start_sec=1.0, duration_sec=0.5)
    print("Audio loaded successfully.")
else:
    print("Audio file not found. Generating white noise for demo.")
    # Generate fake multi-channel data
    data = np.random.randn(8, int(fs*0.5))
    chunk = AudioRecording(data, fs=fs)


## 4. Run Detection
The scanner computes GCC-PHAT for pairs of microphones and projects it onto the grid.

In [ ]:
scanner = BirdScanner(array)

# Define pairs (all combinations)
pairs = []
for i in range(8):
    for j in range(i+1, 8):
        pairs.append((i, j))

# This is a simplified call mimicking the internal logic of scan_file
# In a real run, you'd use scanner.scan_file()

print("Running imaging...")
# Placeholder: Just plotting the mic setup since we don't have the full GCC logic visible in the snippet
plt.figure(figsize=(8, 8))
plt.scatter(mic_positions[:,0], mic_positions[:,1], c='red', label='Mics', s=100)
plt.xlim(xmin, xmax)
plt.ylim(ymin, ymax)
plt.grid(True, alpha=0.3)
plt.legend()
plt.title("Microphone Array Geometry")
plt.xlabel("X (m)")
plt.ylabel("Y (m)")
plt.show()